# cthreads CPU demo: Mandelbrot (pool + Shared + TBuffer)

Compares **pure Python** Mandelbrot with a **multithreaded cthreads** render that uses:

| Piece | Role |
|---|---|
| `ThreadPool` | Fixed workers; many strip jobs without one OS thread per job |
| `Shared[list[int]]` | Cooperative host memory for small per-strip done flags |
| `TBufferI64` | Triple-buffered pixel store: **pointer shared**, no per-pixel list marshal |

Why not one `@Thread` + `list[int]`? Packing/writeback of ~300k ints via per-element
ctypes can dominate runtime and make cthreads look slower than Python.

## What you need

1. `pip install cthreads` (**0.2.1+**)
2. A C++ compiler for the first kernel build (Colab: `g++` via `apt`)

Warm up once (compile + link), then time steady-state runs.

## 1. Setup

In [ ]:
%pip install -q -U "cthreads>=0.2.1" matplotlib
!apt-get -qq install -y g++

: 

## 2. Parameters

Pixels live in a flat buffer of length `width * height`. Work is split into
**row strips** across the pool (Mandelbrot cost varies by region, thus more strips
than workers helps load-balance).

In [ ]:
import os

WIDTH: int    = 640   # image width
HEIGHT: int   = 480   # image height
MAX_ITER: int = 120   # max iterations
X_MIN: float  = -2.0  # x range (lower bound)
X_MAX: float  = 1.0   # x range (upper bound)
Y_MIN: float  = -1.2  # y range (lower bound)
Y_MAX: float  = 1.2   # y range (upper bound)

N: int = WIDTH * HEIGHT # pixel count
WORKERS: int = max(2, (os.cpu_count() or 2)) # OS threads
N_STRIPS: int = WORKERS * 4 # strips (4x workers for load-balance)

print(f"pixels={N}  workers={WORKERS}  strips={N_STRIPS}")

## 3. Pure Python baseline

Same escape-time math; **single-threaded**; writes a plain `list[int]`.

### Why not `threading` for the Python side?

This loop is **CPU-bound pure Python** (tight float/int work, almost no I/O).
CPython has one **GIL** per process: only one thread runs bytecode at a time.

If you split rows across `threading.Thread`s:

- you still serialize on the GIL -> little or no parallel speedup
- you **add** thread create/join, scheduling, and shared-list contention
- so multithreaded Python is often **slower** than one thread here

`multiprocessing` can use cores (separate interpreters) but pays spawn + pickling
the image buffer, which is usually not worth it for this demo size.

**cthreads** is different: `@Thread` kernels run as **compiled C++ off the GIL**,
and `ThreadPool` + `TBufferI64` share native memory without per-pixel Python marshal.

In [ ]:
def mandelbrot_python(
    width: int,
    height: int,
    max_iter: int,
    x_min: float,
    x_max: float,
    y_min: float,
    y_max: float,
    out: list[int],
) -> None:
    for py in range(height):
        for px in range(width):
            x0: float = x_min + (x_max - x_min) * (1.0 * px) / (1.0 * width)
            y0: float = y_min + (y_max - y_min) * (1.0 * py) / (1.0 * height)
            x: float = 0.0
            y: float = 0.0
            i: int = 0
            while i < max_iter:
                if x * x + y * y > 4.0:
                    break
                xt: float = x * x - y * y + x0
                y = 2.0 * x * y + y0
                x = xt
                i = i + 1
            out[py * width + px] = i

## 4. cthreads kernel: strip worker

- `pixels: TBufferI64`: shared native triple buffer (passed by handle; workers
  write **disjoint** row ranges into the current write slot).
- `done: Shared[list[int]]`: small cooperative list on the **pool SharedHost**
  (`done[strip_id] = 1` when a strip finishes).
- Launch many strips via `ThreadPool.group` (pins Shared safely for the wave).
- After all jobs finish, host calls `pixels.publish()` once, then `read_copy()`.

In [ ]:
import cthreads
from cthreads import Thread, Shared, ThreadPool, prepare, load_kernels
from cthreads.sync import TBufferI64

# safety check (usually not required, ensures that the native c++ backend is valid)
if TBufferI64 is None:
    raise RuntimeError("cthreads.sync.TBufferI64 missing — need a CPU/GPU wheel with _ext")


@Thread
def mandelbrot_strip(
    pixels: TBufferI64,
    done: Shared[list[int]],
    strip_id: int,
    y0: int,
    y1: int,
    width: int,
    height: int,
    max_iter: int,
    x_min: float,
    x_max: float,
    y_min: float,
    y_max: float,
) -> None:
    """
    CThreaded mandelbrot function. Runs the same code as the python version,
    execpt that it operates on a local slice of the full img. This way the work is
    shared accross the OS Threads tha the cthreads threadpool launches.
    """
    py: int = y0
    while py < y1:
        px: int = 0
        while px < width:
            x0: float = x_min + (x_max - x_min) * (1.0 * px) / (1.0 * width)
            y0c: float = y_min + (y_max - y_min) * (1.0 * py) / (1.0 * height)
            x: float = 0.0
            y: float = 0.0
            i: int = 0
            while i < max_iter:
                if x * x + y * y > 4.0:
                    break
                xt: float = x * x - y * y + x0
                y = 2.0 * x * y + y0c
                x = xt
                i = i + 1
            pixels[py * width + px] = i
            px = px + 1
        py = py + 1
    done[strip_id] = 1


def strip_ranges(height: int, n_strips: int) -> list[tuple[int, int]]:
    """
    Helper function that splits the flat pixel buffer into N even ranges
    """
    out: list[tuple[int, int]] = []
    for s in range(n_strips):
        y0 = (s * height) // n_strips
        y1 = ((s + 1) * height) // n_strips
        out.append((y0, y1))
    return out


def render_cthreads(
    pool: ThreadPool,
    pixels: TBufferI64,
    n_strips: int,
) -> list[int]:
    """
    Run strip jobs on the pool; publish once; return a Python list snapshot.
    """
    done: list[int] = [0] * n_strips # done flag for the ranges for work distribution
    ranges = strip_ranges(HEIGHT, n_strips) # split the image into N strips
    items = [ # prepare the range wise mandelbrot call arguments
        (
            pixels,
            done,
            s,
            y0,
            y1,
            WIDTH,
            HEIGHT,
            MAX_ITER,
            X_MIN,
            X_MAX,
            Y_MIN,
            Y_MAX,
        )
        for s, (y0, y1) in enumerate(ranges)
    ]
    # submit the work to the cthreads.pool
    group = pool.group(mandelbrot_strip, items)
    # wait for all the strips to finish
    group.results()
    # check that all the strips finished
    if sum(done) != n_strips:
        raise RuntimeError(f"Shared done flags incomplete: {sum(done)}/{n_strips}")
    # retrive the actual result from the TBufferI64
    # (this is GIL free through the internal datastructure and therefore prefered for large data loads)
    pixels.publish()
    return list(pixels.read_copy()) # copy and return

## 5. Warmup, then time both

Warmup compiles the kernel and touches the pool path. Timed runs are steady-state.

In [ ]:
import time

# Pool submit does not auto-prepare; compile + load kernels once.
binary = prepare()
load_kernels(str(binary))

# create the threadpool
pool = ThreadPool(WORKERS).start()
# init the pixel buffer
pixels = TBufferI64(N)

try:
    # --- warmup (first Shared/TBuffer wave; do not time) ---
    _ = render_cthreads(pool, pixels, N_STRIPS)

    # --- timed pure Python ---
    out_py: list[int] = [0] * N
    t0 = time.perf_counter()
    mandelbrot_python(WIDTH, HEIGHT, MAX_ITER, X_MIN, X_MAX, Y_MIN, Y_MAX, out_py)
    t_python = time.perf_counter() - t0

    # --- timed cthreads (pool + Shared + TBuffer) ---
    t0 = time.perf_counter()
    out_native = render_cthreads(pool, pixels, N_STRIPS)
    t_native = time.perf_counter() - t0

    print(f"pure Python:  {t_python * 1000:.1f} ms")
    print(f"cthreads:     {t_native * 1000:.1f} ms")
    if t_native > 0.0:
        print(f"speedup:      {t_python / t_native:.1f}x")

    mismatches = sum(1 for a, b in zip(out_py, out_native) if a != b)
    print(f"pixel mismatches: {mismatches}")
    print(f"Shared done flags: all {N_STRIPS} strips marked complete")
finally:
    pool.stop()

## 6. Visual check

Left / right: Python vs cthreads. Right: timing bars.

In [ ]:
import matplotlib.pyplot as plt

img_py = [out_py[r * WIDTH:(r + 1) * WIDTH] for r in range(HEIGHT)]
img_native = [out_native[r * WIDTH:(r + 1) * WIDTH] for r in range(HEIGHT)]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img_py, cmap="magma", origin="lower")
axes[0].set_title("Pure Python")
axes[0].axis("off")

axes[1].imshow(img_native, cmap="magma", origin="lower")
axes[1].set_title("cthreads pool+Shared+TBuffer")
axes[1].axis("off")

axes[2].bar(
    ["Pure Python", "cthreads"],
    [t_python * 1000.0, t_native * 1000.0],
    color=["#888888", "#2a9d8f"],
)
axes[2].set_ylabel("ms")
axes[2].set_title("Render time")

fig.tight_layout()
plt.show()

## Takeaways

- **`TBufferI64`** — big pixel buffers stay native; pass one handle to every strip job.
  Publish once after the wave, then `read_copy()` for Python/matplotlib.
- **`Shared[list[int]]`** — small cooperative state on the pool’s SharedHost; use
  `pool.group` (or `submit_queue`) so the host stays pinned for the whole wave.
- **`ThreadPool`** — reuses workers; better than one OS thread per strip.
- Avoid timing a single job that packs a huge `list[int]` — marshal will dominate.

Docs: [pools](https://github.com/K-T0BIAS/CThreads/blob/main/docs/guide/pools.md),
[sync / TBuffer](https://github.com/K-T0BIAS/CThreads/blob/main/docs/guide/sync.md),
[Shared marshal](https://github.com/K-T0BIAS/CThreads/blob/main/docs/guide/marshal_and_module.md).